# NeuroLens: paired 2D/3D BraTS learning on Colab T4

This notebook runs the source project on real BraTS 2023 GLI data. It does not fabricate patients or metrics. Store the project folder and dataset in Drive, then set the paths below. Training conditions A, B, and C save to separate directories.

In [ ]:
# 1. Verify that this is a GPU Colab runtime before loading the project.
import importlib.util, sys
if importlib.util.find_spec('torch') is None:
    raise RuntimeError('PyTorch is missing. In Colab choose Runtime > Change runtime type > T4 GPU, then reconnect. This notebook will not install CUDA or replace PyTorch.')
import torch
print('PyTorch:', torch.__version__)
print('CUDA available:', torch.cuda.is_available())
if not torch.cuda.is_available():
    raise RuntimeError('CUDA GPU unavailable. Select an NVIDIA T4 GPU runtime in Colab.')
print(torch.cuda.get_device_name(0))

In [ ]:
# 2. Mount Drive and point to the project folder and BraTS patient directories.
from google.colab import drive
from pathlib import Path
import sys
drive.mount('/content/drive')
PROJECT_DIR = Path('/content/drive/MyDrive/NeuroLens')
DATA_ROOT = Path('/content/drive/MyDrive/BraTS2023_GLI')
if not (PROJECT_DIR / 'train.py').is_file():
    raise FileNotFoundError(f'Copy the NeuroLens project folder to Drive: {PROJECT_DIR}')
if not DATA_ROOT.is_dir():
    raise FileNotFoundError(f'BraTS data folder not found: {DATA_ROOT}')
%cd $PROJECT_DIR
sys.path.insert(0, str(PROJECT_DIR))

In [ ]:
# 3. Check imports. A missing package is reported separately from an installed
# package that fails during import (ABI, transitive dependency, or code error).
import importlib
def import_status(module):
    try:
        importlib.import_module(module)
        return 'available', None
    except ModuleNotFoundError as exc:
        if exc.name == module or module.startswith(str(exc.name) + '.'):
            return 'missing', str(exc)
        return 'import error', f'missing transitive module {exc.name}: {exc}'
    except Exception as exc:
        return 'import error', f'{type(exc).__name__}: {exc}'
required = ['torch', 'numpy', 'nibabel', 'yaml']
optional = ['sklearn', 'matplotlib', 'scipy', 'PIL']
status = {name: import_status(name) for name in required + optional}
for name, result in status.items(): print(name, result)
broken = {k:v for k,v in status.items() if v[0] == 'import error'}
if broken: raise RuntimeError(f'Installed package import/code errors need fixing; do not pip-reinstall blindly: {broken}')
missing_required = [k for k in required if status[k][0] == 'missing']
if 'torch' in missing_required or 'numpy' in missing_required:
    raise RuntimeError(f'Colab runtime is missing core packages {missing_required}; select a standard Colab GPU runtime. Do not install a CUDA stack manually.')
missing_optional = [k for k in optional if status[k][0] == 'missing']
missing_data = [k for k in ('nibabel','yaml') if status[k][0] == 'missing']
pip_names = {'nibabel':'nibabel','yaml':'PyYAML','sklearn':'scikit-learn','matplotlib':'matplotlib','scipy':'scipy','PIL':'Pillow'}
to_install = sorted(set(missing_data + missing_optional))
if to_install:
    import subprocess
    subprocess.check_call([sys.executable, '-m', 'pip', 'install', *[pip_names[x] for x in to_install]])
print('Only absent non-core packages were installed. PyTorch/NumPy were not changed.')

In [ ]:
# 4. Inspect complete cases and patient-level split before starting training.
from neurolens.config import load_config
from neurolens.datasets.brats import discover_patients, split_patients
cfg = load_config('config.yaml')
cfg.data_root = str(DATA_ROOT)
cfg.device = 'cuda'
cfg.mixed_precision = True
cfg.batch_size = max(2, cfg.batch_size)
cfg.validate()
cases = discover_patients(cfg.data_root, cfg.modalities)
train_ids, val_ids = split_patients([x[0] for x in cases], cfg.val_fraction, cfg.seed)
print(f'Complete patients: {len(cases)}; train: {len(train_ids)}; validation: {len(val_ids)}')
print('No image samples or metrics are synthesized by this check.')

In [ ]:
# 5. CPU software smoke test. Temporary random tensors are used only for
# dimension/loss/backprop verification, never for reported experiments.
import subprocess
subprocess.check_call([sys.executable, 'smoke_test.py'])

## Training conditions
A disables cross-dimensional losses, B enables symmetric InfoNCE, and C enables InfoNCE plus embedding consistency. Each gets a separate output directory and starts from the same configured seed. Use patch dimensions divisible by 8 and at least two pairs per batch. Lower `base_channels` or patch size if T4 memory is insufficient. Cell 6 runs all three conditions; edit `CONDITIONS` if you want a subset.

In [ ]:
# 6. Train on the actual BraTS data. This can take a long time.
CONDITIONS = ['A', 'B', 'C']
import subprocess
for condition in CONDITIONS:
    output_dir = f'runs/condition_{condition}'
    cmd = [sys.executable, 'train.py', '--config', 'config.yaml', '--data-root', str(DATA_ROOT), '--condition', condition, '--output-dir', output_dir]
    print('Running:', ' '.join(cmd))
    subprocess.check_call(cmd)

In [ ]:
# 7. Evaluate each completed checkpoint on the deterministic held-out patients.
for condition in CONDITIONS:
    checkpoint = f'runs/condition_{condition}/latest.pt'
    cmd = [sys.executable, 'evaluate.py', '--config', 'config.yaml', '--checkpoint', checkpoint]
    print(f'Condition {condition}')
    subprocess.check_call(cmd)

In [ ]:
# 8. Plot observed training/validation loss curves from actual CSV logs.
from neurolens.evaluation.visualization import plot_training_curves
for condition in CONDITIONS:
    csv_path = f'runs/condition_{condition}/training.csv'
    plot_training_curves(csv_path, f'runs/condition_{condition}/loss_curves.png')
print('Charts use recorded training logs only; unrun conditions have no chart.')